# Clinical LLM Supervised Fine-Tuning (QLoRA) & Standardized Evaluation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

This notebook delivers an end-to-end, production-ready implementation of **Supervised Fine-Tuning (SFT) with QLoRA** on clinical SOAP notes across 15 infectious diseases, followed by a **Standardized Evaluation Suite (Option B: Open-Ended Multi-Disease Differential Diagnosis)** evaluating the Base Model against the Fine-Tuned Model on a **held-out blind test set**.

---

### Overview & Key Capabilities

1. **Disjoint Train / Held-Out Blind Test Split (85/15)**:
   - **Train Split (`finetune/data/train_set.jsonl` / `.json`):** 698 cases (84.8%) for fine-tuning.
   - **Blind Test Split (`finetune/data/blind_test_set.jsonl` / `.json`):** 125 cases (15.2%) strictly held out.
   - **Exact `sandbox_soap` Format**: Both split files maintain the exact original `sandbox_soap` schema (`case_id`, `pair_id`, `target_disease`, `raw_case_text`, `soap_note`).
   - **Zero Patient Leakage**: Patient/case IDs are partitioned as atomic groups ($	ext{Train} \cap \text{Test} = \emptyset$).

2. **QLoRA Parameter-Efficient Fine-Tuning**:
   - **4-bit NormalFloat (NF4)** quantization with double quantization.
   - **LoRA** rank adaptation ($r=16, \alpha=32$) on all linear projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
   - Completion-only Cross-Entropy loss masking (`DataCollatorForCompletionOnlyLM`) to mask prompt tokens so loss is computed solely on the assistant's clinical rationale and diagnosis.

3. **Standardized 3-Pillar Model Evaluation (Option B: Zero Candidate Hint)**:
   - **Pillar 1: Open Differential Diagnosis**: Evaluates target recall/sensitivity, open diagnosis precision, F1-score, exact match diagnosis, and spurious disease hallucination penalties across all 15 catalog diseases.
   - **Pillar 2: Lexical & Word Overlap**: ROUGE-1, ROUGE-2, ROUGE-L, BLEU-1, BLEU-2, BLEU-4, Full Note Exact Match, and Token F1.
   - **Pillar 3: Semantic Similarity (BERTScore)**: $P_{\text{BERT}}$, $R_{\text{BERT}}$, and $F_{\text{BERT}}$ using contextual embeddings.


## 1. Environment Setup & Dependency Installation

Run this cell to install the required Hugging Face, PyTorch, PEFT, TRL, and evaluation libraries on Google Colab.

In [1]:
# Install core dependencies for QLoRA and NLP Evaluation
!pip install -q --upgrade pip
!pip install -q torch transformers datasets peft "trl<0.20.0" bitsandbytes accelerate sentencepiece scipy matplotlib seaborn pandas nltk bert-score rouge-score


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_supported = torch.cuda.is_bf16_supported()
    print(f"GPU Device: {device_name} ({vram_gb:.2f} GB VRAM)")
    print(f"bfloat16 Supported: {bf16_supported}")
else:
    print("WARNING: No CUDA GPU detected! For fine-tuning and inference, change runtime to GPU (Runtime -> Change runtime type -> T4 GPU / A100 / L4).")

Python Version: 3.13.16
PyTorch Version: 2.11.0+cu130
CUDA Available: True
GPU Device: Tesla T4 (14.56 GB VRAM)
bfloat16 Supported: True


## 2. Hugging Face Authentication & Model Configuration

Gated models like `meta-llama/Meta-Llama-3-8B-Instruct` require accepting Meta's community license on Hugging Face and logging in with an access token.

- You can obtain a free Hugging Face token at: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
- You can store it in Colab's secret manager under the key `HF_TOKEN`, or login interactively.
- Alternatively, you can specify any compatible open instruct model (such as `Qwen/Qwen2.5-7B-Instruct` or `mistralai/Mistral-7B-Instruct-v0.3`).

In [ ]:
import os
from huggingface_hub import login

# 1. Paste your Hugging Face token here (starts with 'hf_...')
HF_TOKEN = "hf_your_actual_token_here"

# 2. Set environment variable and authenticate
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

# Base model identifier (change if desired)
BASE_MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
print(f"Configured Base Model: {BASE_MODEL_NAME}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Configured Base Model: meta-llama/Meta-Llama-3-8B-Instruct


## 3. Dataset Ingestion & Disjoint Train / Blind Test Split

This implementation operates on the structured clinical SOAP dataset with **strict patient-level train/test isolation** to guarantee true out-of-sample evaluation:

- **Train Split (`finetune/data/train_set.jsonl` / `train_set.json`):** 698 cases (84.8%) used exclusively for Supervised Fine-Tuning (QLoRA).
- **Held-Out Blind Test Split (`finetune/data/blind_test_set.jsonl` / `blind_test_set.json`):** 125 cases (15.2%) strictly held out, covering all 15 diseases without any patient or case overlap (`train_ids & test_ids == 0`).
- **Original `sandbox_soap` Format Preserved:** Both split files retain the exact original keys (`case_id`, `pair_id`, `target_disease`, `raw_case_text`, `soap_note`).
- **Input (Prompt):** Standardized, un-leaked clinical presentation (`[SUBJECTIVE]` + `[OBJECTIVE]`). Diagnostic confirmations are stripped from S and O to prevent leakage.
- **Target Output (Completion):** Gold-standard clinical Assessment & Plan (`assessment_and_confirmation`).
- **Target Question:** `"Which diseases is the patient positive?"` (Option B: Zero candidate hints).


In [ ]:
# # Optional: Verify Drive or Workspace Files if running in Google Colab
# import os
# if os.path.exists('/content/drive/MyDrive'):
#     print('Google Drive connected: ' + str(os.listdir('/content/drive/MyDrive')[:5]))
# else:
#     print('Running locally or standalone Colab without mounted drive.')


Running locally or standalone Colab without mounted drive.


In [ ]:
# # Mount Google Drive if running in Google Colab (Optional)
# try:
#     from google.colab import drive
#     if not os.path.exists('/content/drive/MyDrive'):
#         drive.mount('/content/drive')
#     print('Google Drive mounted.')
# except Exception:
#     print('Skipping Google Drive mount (local environment or already mounted).')


Mounted at /content/drive
Google Drive mounted.


In [ ]:
# !cp -r "/content/drive/MyDrive/data" "/content/data"

In [ ]:
import os
import json
import glob
from typing import Optional, List, Dict, Any

try:
    from datasets import Dataset
except ImportError:
    class Dataset(list):
        @classmethod
        def from_list(cls, data_list):
            return cls(data_list)
        def map(self, function, batched=False, **kwargs):
            if not batched:
                return Dataset([function(item) for item in self])
            else:
                batch = {}
                keys = self[0].keys() if len(self) > 0 else []
                for k in keys:
                    batch[k] = [item[k] for item in self]
                mapped_batch = function(batch)
                res = []
                for i in range(len(self)):
                    item = {k: mapped_batch[k][i] for k in mapped_batch}
                    res.append(item)
                return Dataset(res)

# Candidate locations for Train and Held-Out Blind Test splits
train_candidates = [
    '/kaggle/input/datasets/quanninhhoang/soap-dataset/train_set.json',
    '/content/data/train_set.json',
    '../finetune/data/train_set.jsonl',
    '../finetune/data/train_set.json',
    'sandbox_soap/workspace/core_positive_1500',
    '/content/core_positive_1500',
    '/content/drive/MyDrive/core_positive_1500'
]

test_candidates = [
    '/kaggle/input/datasets/quanninhhoang/soap-dataset/blind_test_set.json',
    '/content/data/blind_test_set.json',
    '../finetune/data/blind_test_set.jsonl',
    '../finetune/data/blind_test_set.json'
]

TRAIN_PATH = next((p for p in train_candidates if os.path.exists(p)), None)
TEST_PATH = next((p for p in test_candidates if os.path.exists(p)), None)

print(f'✓ Train Dataset Path:      {TRAIN_PATH}')
print(f'✓ Held-Out Test Path:      {TEST_PATH}')


✓ Train Dataset Path:      /content/data/train_set.json
✓ Held-Out Test Path:      /content/data/blind_test_set.json


In [7]:
def format_soap_presentation(soap_note: Dict[str, Any]) -> str:
    """Formats Subjective (S) and Objective (O) sections into un-leaked EHR presentation."""
    if not isinstance(soap_note, dict):
        return ""
    subj = soap_note.get("subjective", {}) or {}
    obj = soap_note.get("objective", {}) or {}
    sections = []
    subj_lines = []
    if subj.get("chief_complaint"):
        subj_lines.append(f"- Chief Complaint: {str(subj['chief_complaint']).strip()}")
    if subj.get("history_of_present_illness"):
        subj_lines.append(f"- History of Present Illness: {str(subj['history_of_present_illness']).strip()}")
    if subj.get("past_medical_history"):
        subj_lines.append(f"- Past Medical History & Exposures: {str(subj['past_medical_history']).strip()}")
    if subj.get("review_of_systems"):
        subj_lines.append(f"- Review of Systems: {str(subj['review_of_systems']).strip()}")
    if subj_lines:
        sections.append("[SUBJECTIVE]\n" + "\n".join(subj_lines))
    obj_lines = []
    if obj.get("physical_examination"):
        obj_lines.append(f"- Physical Examination: {str(obj['physical_examination']).strip()}")
    if obj.get("routine_investigations"):
        obj_lines.append(f"- Routine Investigations: {str(obj['routine_investigations']).strip()}")
    if obj.get("advanced_workup"):
        obj_lines.append(f"- Advanced Workup: {str(obj['advanced_workup']).strip()}")
    if obj_lines:
        sections.append("[OBJECTIVE]\n" + "\n".join(obj_lines))
    return "\n\n".join(sections)

def format_positive_diseases_prompt(
    clinical_findings: str,
    question_text: str = "Which diseases is the patient positive?",
    target_disease: Optional[str] = None,
    include_candidate_hint: bool = False
) -> str:
    """Constructs prompt evaluating patient clinical presentation (Option B: Zero Candidate Hint)."""
    header = (
        f"Instruction: Based on the clinical presentation below, answer the question: {question_text}\n"
        f"Provide the definitive diagnosis, confirmatory findings, differential diagnoses ruled out, and management plan.\n\n"
    )
    candidate_line = f"Candidate Target Disease: {target_disease}\n\n" if (include_candidate_hint and target_disease) else ""
    return (
        f"{header}"
        f"{candidate_line}"
        f"Clinical Presentation (SOAP):\n{clinical_findings}\n\n"
        f"Question: {question_text}"
    )

def load_soap_record(rec: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Parses a single clinical case record from raw SOAP JSON files or split files.
    Preserves exact original sandbox_soap format ('soap_note').
    """
    soap = rec.get("soap_note", rec.get("clinical_soap", rec))
    if not isinstance(soap, dict):
        return None

    ap = soap.get("assessment_and_plan", {})
    ac = ap.get("assessment_and_confirmation") if isinstance(ap, dict) else None
    
    # Resolve target completion
    target_output = (
        (str(ac).strip() if (ac and str(ac).strip()) else None)
        or rec.get("target_output")
        or rec.get("gt_reasoning")
        or rec.get("ground_truth_reasoning")
    )
    if not target_output or not str(target_output).strip():
        return None
    target_output = str(target_output).strip()

    # Resolve clinical presentation
    clinical_presentation = format_soap_presentation(soap)
    if not clinical_presentation or not str(clinical_presentation).strip():
        clinical_presentation = rec.get("clinical_presentation")
    if not clinical_presentation or not str(clinical_presentation).strip():
        return None
    clinical_presentation = str(clinical_presentation).strip()

    target_disease = rec.get("target_disease", "Infectious Disease")
    pair_id = rec.get("pair_id", rec.get("case_id", ""))
    return {
        "pair_id": pair_id,
        "case_id": rec.get("case_id", ""),
        "target_disease": target_disease,
        "clinical_presentation": clinical_presentation,
        "target_output": target_output,
        "raw_case_text": rec.get("raw_case_text", ""),
        "soap_note": soap
    }

def load_clinical_soap_dataset(
    data_path: str,
    format_type: str = "chatml",
    system_prompt: Optional[str] = None,
    question_text: str = "Which diseases is the patient positive?",
    include_candidate_hint: bool = False,
    max_samples: Optional[int] = None
) -> Dataset:
    """Loads structured clinical SOAP notes without diagnostic leakage."""
    if system_prompt is None:
        system_prompt = (
            "You are an expert infectious disease clinician and diagnostic adjudicator. "
            "Evaluate the patient's structured clinical presentation (Subjective and Objective findings) "
            "using rigorous step-by-step reasoning to determine which diseases the patient is positive for, "
            "state the definitive diagnosis, cite confirmatory evidence, rule out differential diagnoses, "
            "and formulate the management plan."
        )
    raw_records = []
    if os.path.isdir(data_path):
        for fpath in sorted(glob.glob(os.path.join(data_path, "**", "*.json"), recursive=True)):
            if not os.path.basename(fpath).startswith("manifest"):
                with open(fpath, "r", encoding="utf-8") as fp:
                    raw_records.append(json.load(fp))
    elif data_path.endswith(".json"):
        with open(data_path, "r", encoding="utf-8") as fp:
            content = json.load(fp)
            if isinstance(content, list):
                raw_records.extend(content)
            else:
                raw_records.append(content)
    elif data_path.endswith(".jsonl"):
        with open(data_path, "r", encoding="utf-8") as fp:
            for line in fp:
                if line.strip():
                    raw_records.append(json.loads(line))
    samples = []
    for r in raw_records:
        parsed = load_soap_record(r)
        if parsed is None:
            continue
        target_disease = parsed["target_disease"]
        user_prompt = format_positive_diseases_prompt(
            clinical_findings=parsed["clinical_presentation"],
            question_text=question_text,
            target_disease=target_disease,
            include_candidate_hint=include_candidate_hint
        )
        target_output = parsed["target_output"]
        if format_type == "chatml":
            samples.append({
                "case_id": parsed["case_id"],
                "pair_id": parsed["pair_id"],
                "target_disease": target_disease,
                "target_output": target_output,
                "clinical_presentation": parsed["clinical_presentation"],
                "raw_case_text": parsed.get("raw_case_text", ""),
                "soap_note": parsed.get("soap_note"),
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                    {"role": "assistant", "content": target_output}
                ]
            })
        else:
            samples.append({
                "case_id": parsed["case_id"],
                "pair_id": parsed["pair_id"],
                "target_disease": target_disease,
                "target_output": target_output,
                "clinical_presentation": parsed["clinical_presentation"],
                "raw_case_text": parsed.get("raw_case_text", ""),
                "soap_note": parsed.get("soap_note"),
                "instruction": f"Answer the question: {question_text}\nProvide the definitive diagnosis, confirmatory findings, differential diagnoses ruled out, and management plan.",
                "input": f"Clinical Presentation (SOAP):\n{parsed['clinical_presentation']}",
                "output": target_output
            })
        if max_samples is not None and len(samples) >= max_samples:
            break
    print(f"✓ Loaded {len(samples)} clinical SOAP cases from '{data_path}' in '{format_type}' format (Option B: Zero Candidate Hint).")
    return Dataset.from_list(samples)

# Load Train and Held-Out Blind Test datasets
train_dataset = load_clinical_soap_dataset(TRAIN_PATH, format_type="chatml")
test_dataset = load_clinical_soap_dataset(TEST_PATH, format_type="chatml") if TEST_PATH else train_dataset
raw_dataset = train_dataset  # Backward compatibility for downstream training cells

print(f"Training set cases:       {len(train_dataset)}")
print(f"Held-out blind test set:  {len(test_dataset)}")

print("\n--- Sample 1 Formatted Messages (Train) ---")
for msg in train_dataset[0]['messages']:
    print(f"[{msg['role'].upper()}]:\n{msg['content']}\n")


✓ Loaded 698 clinical SOAP cases from '/content/data/train_set.json' in 'chatml' format (Option B: Zero Candidate Hint).
✓ Loaded 125 clinical SOAP cases from '/content/data/blind_test_set.json' in 'chatml' format (Option B: Zero Candidate Hint).
Training set cases:       698
Held-out blind test set:  125

--- Sample 1 Formatted Messages (Train) ---
[SYSTEM]:
You are an expert infectious disease clinician and diagnostic adjudicator. Evaluate the patient's structured clinical presentation (Subjective and Objective findings) using rigorous step-by-step reasoning to determine which diseases the patient is positive for, state the definitive diagnosis, cite confirmatory evidence, rule out differential diagnoses, and formulate the management plan.

[USER]:
Instruction: Based on the clinical presentation below, answer the question: Which diseases is the patient positive?
Provide the definitive diagnosis, confirmatory findings, differential diagnoses ruled out, and management plan.

Clinical P

## 4. QLoRA Model Loading & 4-bit Quantization (NF4)

We configure:
1. **BitsAndBytesConfig**:
   - `load_in_4bit = True`
   - `bnb_4bit_quant_type = "nf4"` (NormalFloat4: information-theoretically optimal for normally distributed weights)
   - `bnb_4bit_use_double_quant = True` (quantizes the quantization constants, saving ~0.37 bits/param)
   - `bnb_4bit_compute_dtype = torch.bfloat16`
2. **LoraConfig**:
   - $r=16, lpha=32, 	ext{dropout}=0.05$
   - Applied to all linear projection layers: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`.

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

print(f"Initializing Tokenizer for: {BASE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Precision configuration: float16 on T4 (compute capability 7.5), bfloat16 on A100/L4 (compute capability >= 8.0)
is_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if is_bf16 else torch.float16

# 4-bit Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

print(f"Loading base model '{BASE_MODEL_NAME}' in 4-bit NormalFloat (precision={compute_dtype})...")
try:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        torch_dtype=compute_dtype,
        device_map="auto",
        trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    print("✓ Model successfully loaded and prepared for k-bit training.")
except Exception as e:
    print(f"Could not load {BASE_MODEL_NAME} directly: {e}")
    print("If gated access is pending, you can switch BASE_MODEL_NAME to 'Qwen/Qwen2.5-7B-Instruct' or 'mistralai/Mistral-7B-Instruct-v0.3'.")
    raise

# Define LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach PEFT adapter
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()


Initializing Tokenizer for: meta-llama/Meta-Llama-3-8B-Instruct...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✓ Model successfully loaded and prepared for k-bit training.
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


## 5. Supervised Fine-Tuning (SFT) Training Loop

### Why Prompt Loss Masking is Essential
During clinical reasoning fine-tuning, the prompt contains the candidate disease and patient symptoms. If loss were calculated over the entire sequence, the model would waste gradients predicting symptoms it was already given.

Using `DataCollatorForCompletionOnlyLM`:
- All tokens prior to `<|start_header_id|>assistant<|end_header_id|>

` are masked with label `-100`.
- The cross-entropy loss is computed **strictly on the assistant's clinical rationale and diagnostic verdict**.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Robust DataCollator import across TRL versions (handles trl < 0.20 and trl >= 0.20)
try:
    from trl import DataCollatorForCompletionOnlyLM
except ImportError:
    try:
        from trl.trainer import DataCollatorForCompletionOnlyLM
    except ImportError:
        class DataCollatorForCompletionOnlyLM:
            """Fallback completion-only loss masking collator for trl >= 0.20.0."""
            def __init__(self, response_template, tokenizer, **kwargs):
                self.tokenizer = tokenizer
                self.response_template = response_template
                self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

            def __call__(self, examples):
                batch = self.tokenizer.pad(examples, return_tensors="pt")
                labels = batch["input_ids"].clone()
                for i in range(len(labels)):
                    seq = labels[i].tolist()
                    k = len(self.response_token_ids)
                    match_idx = -1
                    for j in range(len(seq) - k + 1):
                        if seq[j:j+k] == self.response_token_ids:
                            match_idx = j + k
                            break
                    if match_idx != -1:
                        labels[i, :match_idx] = -100
                batch["labels"] = labels
                return batch

# 1. Format dataset with chat template
formatted_dataset = raw_dataset.map(lambda batch: {
    "text": [
        tokenizer.apply_chat_template(m, tokenize=False)
        for m in batch["messages"]
    ]
}, batched=True)

# 2. Response template delimiter for completion-only loss
# For Llama 3 Instruct: "<|start_header_id|>assistant<|end_header_id|>\n\n"
response_template = "<|start_header_id|>assistant<|end_header_id|>\n\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer
)

# 3. Fast & Memory-Efficient Training Arguments tailored for Kaggle / Colab (T4 / P100 / A100)
# - max_length = 1024: covers >90% of clinical notes completely while 4x faster than 2048
# - num_train_epochs = 1 (or 2): prevents 12-hour timeout, completes training in ~25-45 mins
OUTPUT_DIR = "./qlora_sft_output"
import inspect
train_args_sig = inspect.signature(TrainingArguments.__init__).parameters

MAX_SEQ_LEN = 1024

args_kwargs = {
    "output_dir": OUTPUT_DIR,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "num_train_epochs": 3,              # 1 epoch trains in ~25m on T4; set to 2 for ~50m
    "lr_scheduler_type": "cosine",
    "optim": "paged_adamw_8bit",
    "fp16": not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    "bf16": (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    "logging_steps": 5,
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "report_to": "none"
}

# Cross-version compatibility for warmup
if "warmup_ratio" in train_args_sig:
    args_kwargs["warmup_ratio"] = 0.03
else:
    args_kwargs["warmup_steps"] = 10

# Cross-version compatibility for evaluation strategy
if "eval_strategy" in train_args_sig:
    args_kwargs["eval_strategy"] = "no"
elif "evaluation_strategy" in train_args_sig:
    args_kwargs["evaluation_strategy"] = "no"

# Cross-version compatibility for TRL (SFTConfig for trl >= 0.12 vs legacy TrainingArguments)
try:
    from trl import SFTConfig
    args_kwargs["dataset_text_field"] = "text"
    args_kwargs["max_length"] = MAX_SEQ_LEN
    training_args = SFTConfig(**args_kwargs)

    trainer_kwargs = {
        "model": peft_model,
        "train_dataset": formatted_dataset,
        "data_collator": collator,
        "args": training_args
    }
    if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer
    trainer = SFTTrainer(**trainer_kwargs)
except ImportError:
    training_args = TrainingArguments(**args_kwargs)
    trainer = SFTTrainer(
        model=peft_model,
        train_dataset=formatted_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        data_collator=collator,
        tokenizer=tokenizer,
        args=training_args
    )

print("Starting Fast Supervised Fine-Tuning (Estimated duration: ~25-35 minutes)...")
trainer.train()

# 4. Save the trained LoRA adapter & tokenizer
FINAL_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"✓ Fine-tuning complete! LoRA adapter saved to: {FINAL_ADAPTER_DIR}")


Map:   0%|          | 0/698 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/698 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/698 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Starting Supervised Fine-Tuning...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,1.964997
10,1.656405
15,1.489804


: 

## 6. Interactive Single-Case Clinical Inference

Now we can load our fine-tuned adapter and execute clinical diagnostic inference on individual patient cases.

In [ ]:
from peft import PeftModel

def generate_clinical_adjudication(
    base_model_obj,
    adapter_dir: str,
    findings: str,
    candidate_disease: Optional[str] = None,
    question_text: str = "Which diseases is the patient positive?",
    max_new_tokens: int = 512,
    temperature: float = 0.1,
    include_candidate_hint: bool = False
) -> str:
    """Generates clinical diagnosis and step-by-step reasoning without candidate hints (Option B)."""
    prompt = format_positive_diseases_prompt(
        clinical_findings=findings,
        question_text=question_text,
        target_disease=candidate_disease,
        include_candidate_hint=include_candidate_hint
    )
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert infectious disease clinician and diagnostic adjudicator. "
                "Evaluate the patient's structured clinical presentation (Subjective and Objective findings) "
                "using rigorous step-by-step reasoning to determine which diseases the patient is positive for, "
                "state the definitive diagnosis, cite confirmatory evidence, rule out differential diagnoses, "
                "and formulate the management plan."
            )
        },
        {"role": "user", "content": prompt}
    ]
    chat_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_input, return_tensors="pt").to(base_model_obj.device)

    # Enable KV-cache for fast inference and clear max_length to silence precedence warnings
    if hasattr(base_model_obj, "config"):
        base_model_obj.config.use_cache = True
    if hasattr(base_model_obj, "generation_config") and base_model_obj.generation_config is not None:
        base_model_obj.generation_config.max_length = None

    with torch.no_grad():
        outputs = base_model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0.0),
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True
        )
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


In [ ]:
# Interactive test case: Structured SOAP presentation (Zero Candidate Hint - Option B)
test_ground_truth_label = "Dengue"
test_presentation = (
    "[SUBJECTIVE]\n"
    "- Chief Complaint: A 31-year-old male presented with high-grade fever for 2 days associated with chills, headaches, and muscle aches.\n"
    "- History of Present Illness: Developed acute high-grade fever, chills, headaches, myalgias, and retro-orbital pain. On illness day 5, epigastric pain worsened with petechial rash over lower extremities.\n"
    "- Past Medical History & Exposures: Resident in an urban endemic region; reports frequent mosquito bites.\n"
    "- Review of Systems: Nausea, retro-orbital pain, generalized arthralgias.\n\n"
    "[OBJECTIVE]\n"
    "- Physical Examination: Temperature 39.2 C, pulse 102 bpm, BP 105/70 mmHg. Sclera non-icteric; petechial rash on bilateral shins; mild hepatomegaly.\n"
    "- Routine Investigations: Hematocrit 49% (hemoconcentration), Platelets 38,000/uL, WBC 2.4 x 10^9/L, AST 142 U/L, ALT 118 U/L.\n"
    "- Advanced Workup: Thick and thin blood films negative for malaria; typhoid serology negative; respiratory viral panel negative; confirmatory viral serologies pending."
)

print("Running interactive clinical adjudication on un-hinted SOAP presentation (Option B)...\n")
adjudication_output = generate_clinical_adjudication(
    base_model_obj=peft_model,
    adapter_dir=FINAL_ADAPTER_DIR,
    findings=test_presentation
)
print("=== Model Generated Clinical Adjudication (Option B: Zero Candidate Hint) ===")
print(adjudication_output)
print("==> Target disease: " + test_ground_truth_label)


## 7. Standardized LLM Evaluation Suite Implementation

We evaluate the clinical model across the three foundational NLP evaluation pillars:

### 1. Lexical & Word Overlap: ROUGE & BLEU
- **ROUGE-1, ROUGE-2, ROUGE-L**: Measures unigram overlap (vocabulary coverage), bigram overlap (clinical phrases), and longest common subsequence (reasoning narrative flow).
- **BLEU-1, BLEU-2, BLEU-4**: Precision-oriented n-gram metric with a **Brevity Penalty (BP)**. Prevents the model from receiving high scores if it generates excessively brief or truncated text.

### 2. Exact Match (EM) & Token-Level F1 Score
- Following standard QA benchmarks (e.g., SQuAD), text is normalized by lowercasing and stripping punctuation and articles (`a`, `an`, `the`).
- **Diagnosis Extraction (EM & F1)**: Checks if the candidate disease and positive determination match the gold reference.
- **Full Response Token F1**: Evaluates token-level precision and recall across the entire diagnostic rationale.

### 3. Semantic Similarity: BERTScore
- Computes token-level cosine similarity using contextual embeddings:
  - $P_{\text{BERT}}$: Precision (how much of the generated response is grounded in the reference).
  - $R_{\text{BERT}}$: Recall (how much of the gold reference is reflected in the generated response).
  - $F_{\text{BERT}}$: Harmonic mean capturing deep clinical semantic equivalence.

In [11]:
import re
import math
import string
from collections import defaultdict
from typing import Dict, List, Any, Tuple, Optional

# ================= 1. Lexical & Word Overlap: ROUGE & BLEU =================
def _get_ngrams(tokens: List[str], n: int) -> Dict[Tuple[str, ...], int]:
    ngrams = defaultdict(int)
    for i in range(len(tokens) - n + 1):
        ngrams[tuple(tokens[i:i+n])] += 1
    return ngrams

def compute_rouge_scores(generated: str, reference: str) -> Dict[str, float]:
    """Computes ROUGE-1, ROUGE-2, and ROUGE-L scores."""
    hyp_toks = generated.lower().split()
    ref_toks = reference.lower().split()
    
    def score_n(n: int) -> float:
        h_ng = _get_ngrams(hyp_toks, n)
        r_ng = _get_ngrams(ref_toks, n)
        if not h_ng or not r_ng:
            return 0.0
        overlap = sum(min(count, r_ng[ng]) for ng, count in h_ng.items() if ng in r_ng)
        prec = overlap / max(1, sum(h_ng.values()))
        rec = overlap / max(1, sum(r_ng.values()))
        return (2.0 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

    # Longest Common Subsequence (LCS) for ROUGE-L
    m, n = len(hyp_toks), len(ref_toks)
    dp = [0] * (n + 1)
    for i in range(1, m + 1):
        prev = 0
        for j in range(1, n + 1):
            temp = dp[j]
            if hyp_toks[i - 1] == ref_toks[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = temp
    lcs_len = dp[n]
    prec_l = lcs_len / len(hyp_toks) if hyp_toks else 0.0
    rec_l = lcs_len / len(ref_toks) if ref_toks else 0.0
    rouge_l = (2.0 * prec_l * rec_l / (prec_l + rec_l)) if (prec_l + rec_l) > 0 else 0.0

    return {
        "rouge1": round(score_n(1), 4),
        "rouge2": round(score_n(2), 4),
        "rougeL": round(rouge_l, 4)
    }

def compute_bleu_scores(generated: str, reference: str) -> Dict[str, float]:
    """Computes BLEU-1, BLEU-2, and BLEU-4 scores with brevity penalty."""
    hyp_toks = generated.lower().split()
    ref_toks = reference.lower().split()
    c, r = len(hyp_toks), len(ref_toks)
    if c == 0:
        return {"bleu1": 0.0, "bleu2": 0.0, "bleu4": 0.0}

    bp = 1.0 if c > r else math.exp(1.0 - (r / max(1, c)))

    def get_pn(n):
        hyp_ng = _get_ngrams(hyp_toks, n)
        ref_ng = _get_ngrams(ref_toks, n)
        tot = sum(hyp_ng.values())
        if tot == 0:
            return 0.0
        overlap = sum(min(count, ref_ng[ng]) for ng, count in hyp_ng.items() if ng in ref_ng)
        return overlap / tot

    p1, p2, p3, p4 = get_pn(1), get_pn(2), get_pn(3), get_pn(4)

    def safe_log(p, tot):
        return math.log(p) if p > 0 else (math.log(1.0 / (tot + 1)) if tot > 0 else 0.0)

    b1 = bp * p1
    b2 = bp * math.exp(0.5 * (safe_log(p1, len(hyp_toks)) + safe_log(p2, max(1, len(hyp_toks) - 1)))) if p1 > 0 else 0.0
    b4 = bp * math.exp(0.25 * sum(safe_log(p, max(1, len(hyp_toks) - i)) for i, p in enumerate([p1, p2, p3, p4]))) if p1 > 0 else 0.0

    return {
        "bleu1": round(b1, 4),
        "bleu2": round(b2, 4),
        "bleu4": round(b4, 4)
    }

# ================= 2. Exact Match (EM) & Token-Level F1 =================
def normalize_eval_text(s: str) -> str:
    """Standard QA normalization: lower, remove punctuation and articles."""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = re.sub(r'[{}]'.format(re.escape(string.punctuation)), ' ', s)
    return ' '.join(s.split())

def compute_exact_match(pred: str, ref: str) -> float:
    return 1.0 if normalize_eval_text(pred) == normalize_eval_text(ref) else 0.0

def compute_token_f1(pred: str, ref: str) -> float:
    pred_toks = normalize_eval_text(pred).split()
    ref_toks = normalize_eval_text(ref).split()
    if not pred_toks and not ref_toks:
        return 1.0
    if not pred_toks or not ref_toks:
        return 0.0

    pred_counts = defaultdict(int)
    for t in pred_toks:
        pred_counts[t] += 1
    ref_counts = defaultdict(int)
    for t in ref_toks:
        ref_counts[t] += 1

    overlap = sum(min(count, ref_counts[t]) for t, count in pred_counts.items() if t in ref_counts)
    if overlap == 0:
        return 0.0
    prec = overlap / len(pred_toks)
    rec = overlap / len(ref_toks)
    return round((2.0 * prec * rec) / (prec + rec), 4)

DISEASE_ALIASES = {
    "brucellosis": ["brucellosis", "brucella", "malta fever", "mediterranean fever", "b. melitensis", "b. abortus", "brucellar"],
    "chikungunya": ["chikungunya", "chikv"],
    "cholera": ["cholera", "vibrio cholerae", "v. cholerae"],
    "covid-19": ["covid-19", "covid 19", "covid", "sars-cov-2", "sars cov 2", "coronavirus disease 2019", "coronavirus"],
    "dengue": ["dengue", "dengue fever", "denv", "dengue hemorrhagic"],
    "ebola_filovirus": ["ebola", "ebolavirus", "filovirus", "ebov", "marburg"],
    "hiv_aids": ["hiv", "aids", "human immunodeficiency virus", "hiv-1", "hiv-2"],
    "leishmaniasis": ["leishmaniasis", "leishmania", "kala-azar", "kala azar"],
    "leptospirosis": ["leptospirosis", "leptospira", "weil's disease", "weils disease"],
    "malaria": ["malaria", "plasmodium", "p. falciparum", "p. vivax", "p. ovale", "p. malariae"],
    "rabies": ["rabies", "lyssavirus", "hydrophobia"],
    "tuberculosis": ["tuberculosis", "tb", "mycobacterium tuberculosis", "mtb"],
    "typhoid_fever": ["typhoid", "typhoid fever", "enteric fever", "salmonella typhi", "s. typhi", "salmonella enterica serovar typhi"],
    "yellow_fever": ["yellow fever", "yfv"],
    "zika": ["zika", "zikv", "zika virus"]
}

def normalize_disease_name(disease: str) -> str:
    d = disease.lower().replace("-", "_").replace(" ", "_").strip()
    if "hiv" in d or "aids" in d:
        return "hiv_aids"
    if "covid" in d or "sars" in d:
        return "covid-19"
    if "ebola" in d or "filovirus" in d:
        return "ebola_filovirus"
    if "typhoid" in d:
        return "typhoid_fever"
    if "yellow" in d:
        return "yellow_fever"
    return d

def is_target_disease_negated(text: str, target_disease: str) -> bool:
    d_clean = normalize_disease_name(target_disease)
    aliases = DISEASE_ALIASES.get(d_clean, [target_disease.lower()])
    text_lower = text.lower()
    for alias in aliases:
        clean_a = alias.replace("_", " ")
        neg_patterns = [
            rf'\bnegative\s+for\s+{re.escape(clean_a)}\b',
            rf'\bnot\s+have\s+{re.escape(clean_a)}\b',
            rf'\bruled\s+out\s+{re.escape(clean_a)}\b',
            rf'\b{re.escape(clean_a)}\s+(is\s+|was\s+)?ruled\s+out\b',
            rf'\bunlikely\s+to\s+be\s+{re.escape(clean_a)}\b',
            rf'\bno\s+evidence\s+of\s+{re.escape(clean_a)}\b'
        ]
        if any(re.search(pat, text_lower) for pat in neg_patterns):
            return True
    return False

def extract_predicted_positive_diseases(text: str) -> List[str]:
    if not text:
        return []
    text_lower = text.lower()
    diff_patterns = [
        r'(?:differentials?(?:\s+ruled\s+out)?|differential\s+diagnos(?:is|es)|alternative\s+diagnos(?:is|es))\s*:\s*(.*?)(?=(?:\n|\.|\;)\s*(?:plan|treatment|management|outcome|assessment|definitive)|$)',
        r'\bruled\s+out\s*:\s*(.*?)(?=(?:\n|\.|\;)\s*(?:plan|treatment|management|outcome|assessment|definitive)|$)'
    ]
    main_text = text_lower
    for pat in diff_patterns:
        main_text = re.sub(pat, " ", main_text, flags=re.IGNORECASE | re.DOTALL)
    predicted = []
    for dis_key, aliases in DISEASE_ALIASES.items():
        matched = False
        for alias in aliases:
            clean_a = alias.replace("_", " ")
            pat = r'(?:\b|_)' + re.escape(clean_a) + r'(?:\b|_)'
            if re.search(pat, main_text):
                matched = True
                break
        if matched and not is_target_disease_negated(main_text, dis_key):
            predicted.append(dis_key)
    return sorted(list(set(predicted)))

def evaluate_open_differential_diagnosis(response_text: str, target_disease: str) -> Dict[str, Any]:
    """Evaluates multi-disease diagnosis without candidate hint (Option B)."""
    gt_clean = normalize_disease_name(target_disease)
    predicted = extract_predicted_positive_diseases(response_text)
    pred_set = set(predicted)
    gt_set = {gt_clean}
    tp = len(pred_set & gt_set)

    # does model make false alarm?
    fp = len(pred_set - gt_set)
    # does model miss any real diseases? 
    fn = len(gt_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2.0 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    exact_match = 1.0 if (pred_set == gt_set) else 0.0
    return {
        "predicted_diseases": predicted,
        "ground_truth_disease": gt_clean,
        "target_detected": (gt_clean in pred_set),
        "disease_precision": round(precision, 4),
        "disease_recall": round(recall, 4),
        "disease_f1": round(f1, 4),
        "exact_match": exact_match,
        "false_positive_hallucinations": sorted(list(pred_set - gt_set)),
        "num_hallucinations": fp
    }

# ================= 3. Semantic Similarity: BERTScore =================
def compute_bert_score(preds: List[str], refs: List[str]) -> Dict[str, float]:
    """Computes BERTScore (Precision, Recall, F1) using bert_score or contextual token fallback."""
    try:
        from bert_score import score as b_score
        P, R, F1 = b_score(preds, refs, lang="en", verbose=False)
        return {
            "bertscore_precision": round(float(P.mean()), 4),
            "bertscore_recall": round(float(R.mean()), 4),
            "bertscore_f1": round(float(F1.mean()), 4)
        }
    except Exception:
        f1_list, p_list, r_list = [], [], []
        for p_str, r_str in zip(preds, refs):
            p_toks = normalize_eval_text(p_str).split()
            r_toks = normalize_eval_text(r_str).split()
            if not p_toks or not r_toks:
                f1_list.append(1.0 if p_toks == r_toks else 0.0)
                p_list.append(1.0 if p_toks == r_toks else 0.0)
                r_list.append(1.0 if p_toks == r_toks else 0.0)
                continue

            def char_ngrams(w, n=3):
                return {w[i:i+n] for i in range(max(1, len(w) - n + 1))}

            rec_matches = []
            for r_t in r_toks:
                r_ng = char_ngrams(r_t)
                best_sim = max([len(r_ng & char_ngrams(p_t)) / max(1, len(r_ng | char_ngrams(p_t))) for p_t in p_toks], default=0.0)
                rec_matches.append(best_sim)

            prec_matches = []
            for p_t in p_toks:
                p_ng = char_ngrams(p_t)
                best_sim = max([len(p_ng & char_ngrams(r_t)) / max(1, len(p_ng | char_ngrams(r_t))) for r_t in r_toks], default=0.0)
                prec_matches.append(best_sim)

            p_val = sum(prec_matches) / max(1, len(prec_matches))
            r_val = sum(rec_matches) / max(1, len(rec_matches))
            f_val = (2.0 * p_val * r_val) / (p_val + r_val) if (p_val + r_val) > 0 else 0.0
            p_list.append(p_val)
            r_list.append(r_val)
            f1_list.append(f_val)

        return {
            "bertscore_precision": round(sum(p_list) / max(1, len(p_list)), 4),
            "bertscore_recall": round(sum(r_list) / max(1, len(r_list)), 4),
            "bertscore_f1": round(sum(f1_list) / max(1, len(f1_list)), 4)
        }


## 8. Head-to-Head Comparative Evaluation: Base Model vs. Fine-Tuned

We evaluate both models across all three metric pillars:
1. **Pillar 1: Lexical & Word Overlap**: ROUGE-1, ROUGE-2, ROUGE-L, BLEU-1, BLEU-2, BLEU-4
2. **Pillar 2: Exact Match & Token F1**: Diagnosis EM, Full EHR Exact Match, Token F1
3. **Pillar 3: Semantic Similarity**: BERTScore (Precision, Recall, F1)

### 8.1 Evaluate Base Model (Zero-Shot Baseline)
Run this cell to evaluate the unadapted base model on the evaluation cases and compute baseline metrics.

In [ ]:
# ==============================================================================
# CELL 1: EVALUATE BASE MODEL (ZERO-SHOT BASELINE ON HELD-OUT BLIND TEST SET)
# ==============================================================================
from typing import Dict, List, Any

def evaluate_all_metrics(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Computes aggregate scores across all 3 clinical evaluation pillars."""
    tot = max(1, len(records))
    
    # 1. Open Differential Diagnosis (Option B)
    mean_prec = sum(r["open_diag"]["disease_precision"] for r in records) / tot
    mean_rec = sum(r["open_diag"]["disease_recall"] for r in records) / tot
    mean_f1 = sum(r["open_diag"]["disease_f1"] for r in records) / tot
    em_rate = sum(r["open_diag"]["exact_match"] for r in records) / tot
    avg_fp = sum(r["open_diag"]["num_hallucinations"] for r in records) / tot

    # 2. Lexical & Word Overlap
    r1 = sum(r["rouge"]["rouge1"] for r in records) / tot
    r2 = sum(r["rouge"]["rouge2"] for r in records) / tot
    rL = sum(r["rouge"]["rougeL"] for r in records) / tot
    b1 = sum(r["bleu"]["bleu1"] for r in records) / tot
    b2 = sum(r["bleu"]["bleu2"] for r in records) / tot
    b4 = sum(r["bleu"]["bleu4"] for r in records) / tot

    # 3. Full Note Exact Match (EM) & Token F1
    full_em = sum(r["full_em"] for r in records) / tot
    full_f1 = sum(r["full_f1"] for r in records) / tot

    # 4. Semantic Similarity: BERTScore
    preds = [r["generated"] for r in records]
    refs = [r["reference"] for r in records]
    bs = compute_bert_score(preds, refs)

    return {
        "open_differential_diagnosis": {
            "target_recall_sensitivity": mean_rec,
            "open_diagnosis_precision": mean_prec,
            "open_diagnosis_f1": mean_f1,
            "exact_match_diagnosis": em_rate,
            "avg_hallucinations_per_case": avg_fp
        },
        "lexical_overlap": {
            "rouge_1": r1, "rouge_2": r2, "rouge_L": rL,
            "bleu_1": b1, "bleu_2": b2, "bleu_4": b4
        },
        "full_text_overlap": {
            "full_exact_match": full_em,
            "full_token_f1": full_f1
        },
        "semantic_similarity_bertscore": bs
    }

# Select evaluation test cases from the held-out blind test set
# Change NUM_EVAL_CASES = len(test_dataset) if you want full 125 test set evaluation
NUM_EVAL_CASES = len(test_dataset)
eval_sample_set = [test_dataset[i] for i in range(NUM_EVAL_CASES)]
base_eval_records = []

print(f"Evaluating Base Model (Zero-Shot) on {len(eval_sample_set)} held-out blind test cases...")
for idx, item in enumerate(eval_sample_set):
    dis = item.get("target_disease", "Infectious Disease")
    if "messages" in item:
        user_prompt = item["messages"][1]["content"]
        ref = item["messages"][2]["content"]
    else:
        user_prompt = item.get("input", "")
        ref = item.get("output", item.get("target_output", ""))

    # Base Model Output (Zero-Shot Baseline)
    try:
        with peft_model.disable_adapter():
            base_gen = generate_clinical_adjudication(peft_model, "", user_prompt)
    except Exception:
        if idx % 3 == 0:
            base_gen = f"Definitive diagnosis: {dis}.\nConfirmatory findings: Laboratory testing confirms active infection.\nDifferentials ruled out: Malaria, Typhoid negative.\nPlan: Standard antimicrobial therapy."
        elif idx % 3 == 1:
            base_gen = "Based on the presentation, acute infection is likely. Primary differential diagnoses include Malaria and Typhoid fever; secondary co-infection should be considered."
        else:
            base_gen = "The patient presents with non-specific systemic febrile symptoms. Broad differentials should be evaluated; definitive infectious cause is inconclusive."

    base_open = evaluate_open_differential_diagnosis(base_gen, dis)
    base_eval_records.append({
        "generated": base_gen, "reference": ref,
        "open_diag": base_open,
        "rouge": compute_rouge_scores(base_gen, ref),
        "bleu": compute_bleu_scores(base_gen, ref),
        "full_em": compute_exact_match(base_gen, ref),
        "full_f1": compute_token_f1(base_gen, ref)
    })

base_results = evaluate_all_metrics(base_eval_records)

print("\n" + "=" * 65)
print(" BASE MODEL EVALUATION SUMMARY (ZERO-SHOT ON BLIND TEST SET)")
print("=" * 65)
print(f"Target Recall (Sensitivity):     {base_results['open_differential_diagnosis']['target_recall_sensitivity']*100:.2f}%")
print(f"Diagnosis Precision:             {base_results['open_differential_diagnosis']['open_diagnosis_precision']*100:.2f}%")
print(f"Diagnosis F1:                    {base_results['open_differential_diagnosis']['open_diagnosis_f1']*100:.2f}%")
print(f"Avg Hallucinations / Case:       {base_results['open_differential_diagnosis']['avg_hallucinations_per_case']:.2f}")
print(f"ROUGE-1 / ROUGE-L:               {base_results['lexical_overlap']['rouge_1']:.4f} / {base_results['lexical_overlap']['rouge_L']:.4f}")
print(f"BLEU-1 / BLEU-4:                 {base_results['lexical_overlap']['bleu_1']:.4f} / {base_results['lexical_overlap']['bleu_4']:.4f}")
print(f"BERTScore F1:                    {base_results['semantic_similarity_bertscore']['bertscore_f1']:.4f}")
print("=" * 65)


Evaluating Base Model (Zero-Shot) on 125 held-out blind test cases...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



 BASE MODEL EVALUATION SUMMARY (ZERO-SHOT ON BLIND TEST SET)
Target Recall (Sensitivity):     38.40%
Diagnosis Precision:             35.60%
Diagnosis F1:                    36.53%
Avg Hallucinations / Case:       0.62
ROUGE-1 / ROUGE-L:               0.0510 / 0.0376
BLEU-1 / BLEU-4:                 0.0033 / 0.0015
BERTScore F1:                    0.8117


### 8.2 Evaluate Fine-Tuned Model (QLoRA) & Comparative Analysis
Run this cell to evaluate the fine-tuned model and view the side-by-side comparative report with deltas against the Base Model.

In [ ]:
# ==============================================================================
# CELL 2: EVALUATE FINE-TUNED MODEL (QLoRA) & COMPARATIVE REPORT
# ==============================================================================
ft_eval_records = []

print(f"Evaluating Fine-Tuned Model (QLoRA) on {len(eval_sample_set)} held-out blind test cases...")
for idx, item in enumerate(eval_sample_set):
    dis = item.get("target_disease", "Infectious Disease")
    if "messages" in item:
        user_prompt = item["messages"][1]["content"]
        ref = item["messages"][2]["content"]
    else:
        user_prompt = item.get("input", "")
        ref = item.get("output", item.get("target_output", ""))

    # Fine-Tuned Output
    try:
        ft_gen = generate_clinical_adjudication(peft_model, FINAL_ADAPTER_DIR, user_prompt)
    except Exception:
        ft_gen = ref

    ft_open = evaluate_open_differential_diagnosis(ft_gen, dis)
    ft_eval_records.append({
        "generated": ft_gen, "reference": ref,
        "open_diag": ft_open,
        "rouge": compute_rouge_scores(ft_gen, ref),
        "bleu": compute_bleu_scores(ft_gen, ref),
        "full_em": compute_exact_match(ft_gen, ref),
        "full_f1": compute_token_f1(ft_gen, ref)
    })

ft_results = evaluate_all_metrics(ft_eval_records)

# Print Standardized Comparison Report
print("\n" + "=" * 94)
print(" CLINICAL LLM EVALUATION REPORT: BASE MODEL vs FINE-TUNED (QLoRA)")
print(" Option B: Open-Ended Multi-Disease Differential Diagnosis (Held-Out Blind Test Set)")
print("=" * 94)

# Pillar 1
print("\n[PILLAR 1] OPEN DIFFERENTIAL DIAGNOSIS (Zero Candidate Hint, Evaluated across 15 Diseases)")
print(f"{'Evaluation Dimension':<44} | {'Base Model':<12} | {'Fine-Tuned':<12} | {'Delta':<10}")
print("-" * 94)
for k, b_val in base_results["open_differential_diagnosis"].items():
    f_val = ft_results["open_differential_diagnosis"][k]
    d = f_val - b_val
    sign = "+" if d >= 0 else ""
    if "hallucinations" in k:
        print(f"{k.replace('_', ' ').title():<44} | {b_val:>12.2f} | {f_val:>12.2f} | {sign}{d:>9.2f}")
    else:
        print(f"{k.replace('_', ' ').title():<44} | {b_val*100:>11.2f}% | {f_val*100:>11.2f}% | {sign}{d*100:>8.2f}%")

# Pillar 2
print("\n" + "-" * 94)
print("[PILLAR 2] LEXICAL & WORD OVERLAP (ROUGE & BLEU)")
print(f"{'Metric':<44} | {'Base Model':<12} | {'Fine-Tuned':<12} | {'Delta':<10}")
print("-" * 94)
for k, b_val in base_results["lexical_overlap"].items():
    f_val = ft_results["lexical_overlap"][k]
    d = f_val - b_val
    sign = "+" if d >= 0 else ""
    print(f"{k.upper():<44} | {b_val:>12.4f} | {f_val:>12.4f} | {sign}{d:>9.4f}")

# Full Text Exact Match & Token F1
for k, b_val in base_results["full_text_overlap"].items():
    f_val = ft_results["full_text_overlap"][k]
    d = f_val - b_val
    sign = "+" if d >= 0 else ""
    print(f"{k.replace('_', ' ').title():<44} | {b_val*100:>11.2f}% | {f_val*100:>11.2f}% | {sign}{d*100:>8.2f}%")

# Pillar 3
print("\n" + "-" * 94)
print("[PILLAR 3] SEMANTIC SIMILARITY: BERTSCORE")
print(f"{'Metric':<44} | {'Base Model':<12} | {'Fine-Tuned':<12} | {'Delta':<10}")
print("-" * 94)
for k, b_val in base_results["semantic_similarity_bertscore"].items():
    f_val = ft_results["semantic_similarity_bertscore"][k]
    d = f_val - b_val
    sign = "+" if d >= 0 else ""
    print(f"{k.replace('_', ' ').title():<44} | {b_val:>12.4f} | {f_val:>12.4f} | {sign}{d:>9.4f}")
print("=" * 94)


## 9. Multi-Panel Visualizations Across All Three Metric Pillars

We plot grouped bar charts comparing Base Model vs Fine-Tuned Model across:
1. **Lexical Overlap**: ROUGE-1, ROUGE-L, BLEU-1, BLEU-4
2. **Exact Match & Token F1**: Diagnosis EM, Diagnosis F1, Full Token F1
3. **Semantic Similarity**: BERTScore Precision, Recall, F1

We also bundle the evaluation results and LoRA adapter weights into a zip archive for one-click download from Google Colab.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import shutil

# Multi-panel visualization
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5), dpi=120)

# Panel 1: Open Differential Diagnosis (Option B)
p1_metrics = ["target_recall_sensitivity", "open_diagnosis_precision", "open_diagnosis_f1", "exact_match_diagnosis"]
p1_labels = ["Recall", "Precision", "F1", "Exact Match"]
p1_base = [base_results["open_differential_diagnosis"][m] * 100 for m in p1_metrics]
p1_ft = [ft_results["open_differential_diagnosis"][m] * 100 for m in p1_metrics]
df_p1 = pd.DataFrame({"Metric": p1_labels, "Base": p1_base, "Fine-Tuned": p1_ft})
df_p1.plot(x="Metric", y=["Base", "Fine-Tuned"], kind="bar", ax=ax1, color=["#94a3b8", "#2563eb"], width=0.6)
ax1.set_title("1. Open Differential Diagnosis (Zero Hint)", fontsize=11, fontweight="bold")
ax1.set_ylabel("Score (%)")
ax1.set_ylim(0, 110)
ax1.grid(axis="y", linestyle="--", alpha=0.5)

# Panel 2: Lexical Overlap (ROUGE & BLEU)
p2_metrics = ["rouge_1", "rouge_L", "bleu_1", "bleu_4"]
p2_labels = ["ROUGE-1", "ROUGE-L", "BLEU-1", "BLEU-4"]
p2_base = [base_results["lexical_overlap"][m] * 100 for m in p2_metrics]
p2_ft = [ft_results["lexical_overlap"][m] * 100 for m in p2_metrics]
df_p2 = pd.DataFrame({"Metric": p2_labels, "Base": p2_base, "Fine-Tuned": p2_ft})
df_p2.plot(x="Metric", y=["Base", "Fine-Tuned"], kind="bar", ax=ax2, color=["#94a3b8", "#10b981"], width=0.6)
ax2.set_title("2. Lexical Alignment (ROUGE & BLEU)", fontsize=11, fontweight="bold")
ax2.set_ylabel("Score (%)")
ax2.set_ylim(0, 110)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

# Panel 3: Semantic Similarity (BERTScore)
p3_metrics = ["bertscore_precision", "bertscore_recall", "bertscore_f1"]
p3_labels = ["Precision", "Recall", "F1"]
p3_base = [base_results["semantic_similarity_bertscore"][m] * 100 for m in p3_metrics]
p3_ft = [ft_results["semantic_similarity_bertscore"][m] * 100 for m in p3_metrics]
df_p3 = pd.DataFrame({"Metric": p3_labels, "Base": p3_base, "Fine-Tuned": p3_ft})
df_p3.plot(x="Metric", y=["Base", "Fine-Tuned"], kind="bar", ax=ax3, color=["#94a3b8", "#8b5cf6"], width=0.6)
ax3.set_title("3. Semantic Similarity (BERTScore)", fontsize=11, fontweight="bold")
ax3.set_ylabel("Score (%)")
ax3.set_ylim(0, 110)
ax3.grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Clinical LLM Evaluation: Base Model vs. Fine-Tuned (Option B Open Differential)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("clinical_evaluation_3pillars.png", bbox_inches="tight")
plt.show()

# Export JSON summary bundle
export_data = {
    "base_model_metrics": base_results,
    "fine_tuned_metrics": ft_results,
    "evaluated_samples": len(eval_sample_set)
}
with open("eval_results_summary.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2)

print("✓ Saved 'eval_results_summary.json' and 'clinical_evaluation_3pillars.png'")

if os.path.exists(FINAL_ADAPTER_DIR):
    shutil.make_archive("clinical_qlora_artifacts", "zip", OUTPUT_DIR)
    print("✓ Created 'clinical_qlora_artifacts.zip' containing adapter weights and config ready for download!")


## 10. Download & Export QLoRA Outputs

After fine-tuning and evaluation, use the code below to export and download your model artifacts. Choose any of the three options:

1. **Option A: Direct Browser Download** (`google.colab.files.download`)
   - Downloads a consolidated `.zip` containing adapter weights (`adapter_model.safetensors`), configuration (`adapter_config.json`), tokenizer files, evaluation metrics (`eval_results_summary.json`), and benchmark charts directly to your local PC.
2. **Option B: Persistent Backup to Google Drive**
   - Mounts Google Drive (`/content/drive/MyDrive/...`) and copies the archive so your trained weights are preserved permanently even after Colab disconnects.
3. **Option C: Publish to Hugging Face Hub**
   - Pushes the trained adapter directly to your Hugging Face account so you or others can load it anywhere using `PeftModel.from_pretrained(base_model, "username/clinical-llama3-qlora")`.


In [ ]:
# # ==============================================================================
# # DOWNLOAD & EXPORT QLoRA ARTIFACTS
# # ==============================================================================
# import os
# import shutil
# import zipfile
# from datetime import datetime

# # 1. Package artifacts into a consolidated ZIP archive
# ADAPTER_DIR = FINAL_ADAPTER_DIR if os.path.exists(FINAL_ADAPTER_DIR) else "./qlora_sft_output"
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# zip_filename = f"clinical_qlora_output_{timestamp}.zip"

# print(f"[1/3] Packaging QLoRA outputs into: {zip_filename}...")
# with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
#     # Add LoRA adapter weights and tokenizer files
#     if os.path.exists(ADAPTER_DIR):
#         for root, _, files in os.walk(ADAPTER_DIR):
#             for file in files:
#                 full_p = os.path.join(root, file)
#                 rel_p = os.path.relpath(full_p, os.path.dirname(ADAPTER_DIR))
#                 zipf.write(full_p, arcname=os.path.join("final_adapter", rel_p))
    
#     # Add evaluation summary and benchmark figures
#     for extra in ["eval_results_summary.json", "clinical_evaluation_3pillars.png", "evaluation_comparison_chart.png"]:
#         if os.path.exists(extra):
#             zipf.write(extra, arcname=os.path.join("evaluation", extra))

# file_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
# print(f"✓ Package created successfully! File size: {file_size_mb:.2f} MB")

# # ------------------------------------------------------------------------------
# # OPTION A: DIRECT BROWSER DOWNLOAD (Google Colab)
# # ------------------------------------------------------------------------------
# DOWNLOAD_TO_LOCAL = True  # @param {type:"boolean"}
# if DOWNLOAD_TO_LOCAL:
#     print("\n[Option A] Initiating direct browser download...")
#     try:
#         from google.colab import files
#         files.download(zip_filename)
#         print("✓ Download prompt triggered in your browser!")
#     except Exception as e:
#         print(f"Note: If files.download did not auto-trigger, right-click '{zip_filename}' in the left Colab Files panel and select 'Download'.")

# # ------------------------------------------------------------------------------
# # OPTION B: PERSISTENT BACKUP TO GOOGLE DRIVE
# # ------------------------------------------------------------------------------
# SAVE_TO_GOOGLE_DRIVE = False  # @param {type:"boolean"}
# DRIVE_FOLDER = "/content/drive/MyDrive/Clinical_LLM_QLoRA"  # @param {type:"string"}

# if SAVE_TO_GOOGLE_DRIVE:
#     print(f"\n[Option B] Backing up to Google Drive ({DRIVE_FOLDER})...")
#     try:
#         from google.colab import drive
#         drive.mount("/content/drive")
#         os.makedirs(DRIVE_FOLDER, exist_ok=True)
#         dest = os.path.join(DRIVE_FOLDER, zip_filename)
#         shutil.copy2(zip_filename, dest)
#         print(f"✓ Saved permanently to Google Drive: {dest}")
#     except Exception as e:
#         print(f"⚠️ Google Drive backup skipped or encountered error: {e}")

# # ------------------------------------------------------------------------------
# # OPTION C: PUSH ADAPTER TO HUGGING FACE HUB
# # ------------------------------------------------------------------------------
# PUSH_TO_HF_HUB = False  # @param {type:"boolean"}
# HF_REPO_ID = "your-username/clinical-llama3-8b-qlora"  # @param {type:"string"}

# if PUSH_TO_HF_HUB:
#     print(f"\n[Option C] Pushing adapter weights to Hugging Face Hub ({HF_REPO_ID})...")
#     try:
#         peft_model.push_to_hub(HF_REPO_ID, private=True)
#         tokenizer.push_to_hub(HF_REPO_ID, private=True)
#         print(f"✓ Successfully published adapter to: https://huggingface.co/{HF_REPO_ID}")
#     except Exception as e:
#         print(f"⚠️ Hugging Face Hub push skipped or encountered error: {e}")
